# Chapter 7: Overfitting, stress test and regularization criteria
This chapter aims to show how wrong dataset can pollute the solution and presents some strategy to mitigate the problem:
- *Batch weighting/resampling:* Act on the batch in term of weights and/or resampling to ensure capability to handle unbalanced data.
- *Dropout regularization:* Usage of Dropout to prevent early overfitting and increase the probability to reach an optimal solution at the cost of longer convergence time.

In [ ]:
# Libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    precision_score,
    recall_score
)

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader, WeightedRandomSampler, Sampler


path_pardir = Path(os.getcwd()).parent
path_data = os.path.join(path_pardir, 'Data')

# S7-1: Classification model
We want to introduce an elementary classification model on the provided dataset.

First we define the device:

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu" # for this tutorial we force cpu
print("Device:", device)

Read the existing database and subdivide it in training/validation/test:

In [ ]:
chosen_df = "df_classification_large.csv"
chosen_unbalanced_df = "df_classification_unbalanced_full.csv"
chosen_heavy_unbalanced_df = "df_classification_heavy_unbalanced_full.csv"

full_df = pd.read_csv(os.path.join(path_data, chosen_df))

def create_unbalanced(df, class_col="label", class_to_reduce=1, fraction=0.1, seed=42):
    class_mask = df[class_col] == class_to_reduce
    other_mask = ~class_mask
    reduced_class = df[class_mask].sample(frac=fraction, random_state=seed)
    other_classes = df[other_mask]
    unbalanced_df = pd.concat([reduced_class, other_classes]).reset_index(drop=True)
    return unbalanced_df


# Standard dataset
training_df = full_df[full_df["split_label"]==0]
training_unbalanced_df = create_unbalanced(training_df, class_col="label", class_to_reduce=1, fraction=0.01)
training_heavy_unbalanced_df = create_unbalanced(training_df, class_col="label", class_to_reduce=1, fraction=0.002)

validation_df = full_df[full_df["split_label"]==1]
test_df = full_df[full_df["split_label"]==2]

# Save them
df_unbalanced_full = pd.concat([
    training_unbalanced_df,
    validation_df,
    test_df
]).reset_index(drop=True)

df_unbalanced_full.to_csv(
    os.path.join(path_data, chosen_unbalanced_df),
    index=False
)

# Heavy unbalanced version
df_heavy_unbalanced_full = pd.concat([
    training_heavy_unbalanced_df,
    validation_df,
    test_df
]).reset_index(drop=True)

df_heavy_unbalanced_full.to_csv(
    os.path.join(path_data, chosen_heavy_unbalanced_df),
    index=False
)

print("Dataframe length:", len(full_df))
print("Training length:", len(training_df))
print("Training unbalanced length:", len(training_unbalanced_df))
print("Training heavy unbalanced length:", len(training_heavy_unbalanced_df))
print("Validation length:", len(validation_df))
print("Test length:", len(test_df))

training_heavy_unbalanced_df.head(80)

Show the scatterplots of the training and test dataframe:

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,4))

# Scatter plot
sns.scatterplot(data=training_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[0])
sns.scatterplot(data=test_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[1])

Same for unbalanced & heavy unbalanced

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,4))
sns.scatterplot(data=training_unbalanced_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[0])
sns.scatterplot(data=test_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[1])

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,4))
sns.scatterplot(data=training_heavy_unbalanced_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[0])
sns.scatterplot(data=test_df, x="x_range", y="y_range", hue="label", alpha = 0.8, ax = axs[1])

Define a custom dataset that returns a feature vector (in this case (x,y)) and a label (seconded as torch.long()):

In [ ]:
# Dataset definition:
class CustomDataset(Dataset):
    def __init__(self, annotations_file, path_data, split_label=None):
        if split_label!=None:
            self.df = pd.read_csv(os.path.join(path_data, annotations_file))
            self.df = self.df[self.df['split_label']==split_label]
        else:
            self.df = pd.read_csv(os.path.join(path_data, annotations_file))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x_range = float(self.df.iloc[idx]['x_range'])
        y_range = float(self.df.iloc[idx]['y_range'])
        features = torch.tensor([x_range, y_range], dtype=torch.float32)
        
        label = self.df.iloc[idx]['label']

        return features, torch.tensor(label, dtype=torch.long)

# Initialize Training dataset and show outputs and properties:
training_dataset = CustomDataset(annotations_file=chosen_df, path_data=path_data, split_label=0)
training_unbalanced_dataset = CustomDataset(annotations_file=chosen_unbalanced_df, path_data=path_data, split_label=0)
training_heavy_unbalanced_dataset = CustomDataset(annotations_file=chosen_heavy_unbalanced_df, path_data=path_data, split_label=0)

index_example = 27
print(f"Output example ([x_range,y_range],label)=({training_dataset.__getitem__(index_example)}) with index {index_example}.")

# Initialize validation and test datasets:
validation_dataset = CustomDataset(annotations_file=chosen_df, path_data=path_data, split_label=1)
test_dataset = CustomDataset(annotations_file=chosen_df, path_data=path_data, split_label=2)

Define the dataloaders from the CustomDataset:

In [ ]:
# Size of each batch
batch_size = 5 # POSSIBLE MISTAKE: batch size too small

# Define 3 DataLoaders: Training/Validation/Test 
training_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=False) # POSSIBLE MISTAKE: Shuffle == False
training_unbalanced_dataloader = DataLoader(training_unbalanced_dataset, batch_size=batch_size, shuffle=False)
training_heavy_unbalanced_dataloader = DataLoader(training_heavy_unbalanced_dataset, batch_size=batch_size, shuffle=False)

validation_dataloader = DataLoader(validation_dataset, batch_size=1, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

The model is defined as a sequential PyTorch model with 2 inputs (x,y) and 2 outputs (class 0, class 1). The two outputs are returned as "logits", intended to be real-valued numbers.

In [ ]:
# Define the model:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

model_unbalanced = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

model_heavy_unbalanced = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

# Move the model to device (GPU):
model.to(device)
model_unbalanced.to(device)
model_heavy_unbalanced.to(device)

# Convert weights type to float:
model = model.float()
model_unbalanced = model_unbalanced.float()
model_heavy_unbalanced = model_heavy_unbalanced.float()

The loss for (multi-class) classification is the Cross Entropy Loss. In pytorch, it can be defined as a softmax layer followed by a logarithmic transformation and an NLL loss, or it can be computed directly using the CrossEntropyLoss. This second approach involves a computational trick to stabilise the backpropagation.

In [ ]:
# Loss function (Cross entropy loss):
loss_fn = nn.CrossEntropyLoss() # It applies log_softmax and NLL directly

# Optimizer (Adam): 
learning_rate = 1e-2 # POSSIBLE MISTAKE
optimizers = {
    "standard": optim.Adam(model.parameters(), lr=learning_rate),
    "unbalanced": optim.Adam(model_unbalanced.parameters(), lr=learning_rate),
    "heavy_unbalanced": optim.Adam(model_heavy_unbalanced.parameters(), lr=learning_rate)
}

Training cycle:

In [ ]:
# Model info
models_info = [
    {
        "name": "standard",
        "model": model,
        "dataloader": training_dataloader  # training standard
    },
    {
        "name": "unbalanced",
        "model": model_unbalanced,
        "dataloader": training_unbalanced_dataloader  # training 1% class 1
    },
    {
        "name": "heavy_unbalanced",
        "model": model_heavy_unbalanced,
        "dataloader": training_heavy_unbalanced_dataloader  # training 0.2% class 1
    }
]

# Number of training epochs:
n_epochs = 30

all_training_losses = {}
all_validation_losses  = {}

for info in models_info:
    print(f"\nTraining model: {info['name']}")
    
    current_model = info['model']
    current_dataloader = info['dataloader']
    current_optimizer = optimizers[info['name']] 

    training_loss_list = []
    validation_loss_list = []
    
    # Iterate over the epochs:
    for epoch in range(n_epochs):
    
        # Set the model to Training mode: This interacts with certain kind of network layers (such as Dropout layers)
        current_model.train()  
    
        # Temporary variable to store the loss on the whole epoch as a convergence metric
        running_loss = 0.0
    
        # Iterate on the whole dataset using the dataloader.
        for xy_input, label_gt in current_dataloader:
            # Load inputs ad move to device (GPU)
            xy_input, label_gt = xy_input.to(device), label_gt.to(device)
    
            # Clear previous gradients
            current_optimizer.zero_grad()  
            
            # Forward pass (model calls)
            logits_model = current_model(xy_input)  
    
            # Compute loss (supervised case)
            loss = loss_fn(logits_model, label_gt)  
            
            # Backpropagation 
            loss.backward()  
    
            # Update parameters (optimization step)
            current_optimizer.step()  
    
            # Update running loss as convergence metric
            running_loss += loss.item()
            
        ## Calculate loss on validation as an additional metric to evaluate overfitting
        # Set the model to Evaluation mode:
        current_model.eval()
    
        # Temporary variable to store the validation loss:
        running_val_loss = 0.0
        
        # Deactivate gradient computation
        with torch.no_grad():
            for xy_input, label_gt in validation_dataloader:
                # Load inputs ad move to device (GPU)
                xy_input, label_gt = xy_input.to(device), label_gt.to(device)
    
                # Forward pass (model calls)
                logits_model = current_model(xy_input)
    
                # Compute loss (supervised case)
                loss = loss_fn(logits_model, label_gt)
    
                # Update validation running loss as convergence metric
                running_val_loss += loss.item()
                
        # Average epoch loss
        epoch_training_loss = running_loss / len(current_dataloader)
        epoch_validation_loss = running_val_loss / len(validation_dataloader)
    
        # Append the losses to the list:
        training_loss_list.append(epoch_training_loss)
        validation_loss_list.append(epoch_validation_loss)
                
    
        # Convergence metric
        print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_training_loss:.4f}, Val. Loss: {epoch_validation_loss:.4f}")
        
    # Salva i risultati nel dizionario
    all_training_losses[info['name']] = training_loss_list
    all_validation_losses[info['name']] = validation_loss_list

See confusion matrices:

In [ ]:
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xy_input, label_gt in dataloader:
            xy_input = xy_input.to(device)
            label_gt = label_gt.to(device)

            logits = model(xy_input)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_gt.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

def normalize_cm(cm):
    cm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = cm / row_sums
    return cm_norm

# Standard
y_true_std, y_pred_std = get_predictions(model, test_dataloader, device)
cm_std = confusion_matrix(y_true_std, y_pred_std)

# Unbalanced
y_true_unb, y_pred_unb = get_predictions(model_unbalanced, test_dataloader, device)
cm_unb = confusion_matrix(y_true_unb, y_pred_unb)

# Heavy unbalanced
y_true_hunb, y_pred_hunb = get_predictions(model_heavy_unbalanced, test_dataloader, device)
cm_hunb = confusion_matrix(y_true_hunb, y_pred_hunb)


# Normalize
cm_std_norm  = normalize_cm(cm_std)
cm_unb_norm  = normalize_cm(cm_unb)
cm_hunb_norm = normalize_cm(cm_hunb)

# Figures
fig, axes = plt.subplots(1, 3, figsize=(18,5))

cms = [cm_std_norm, cm_unb_norm, cm_hunb_norm]
titles = ["Standard", "Unbalanced (1%)", "Heavy Unbalanced (0.2%)"]

for ax, cm, title in zip(axes, cms, titles):
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        vmin=0,
        vmax=1,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

Evaluate the usage of specific metrics:

In [ ]:
def compute_metrics(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xy_input, label_gt in dataloader:
            xy_input = xy_input.to(device)
            label_gt = label_gt.to(device)

            logits = model(xy_input)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_gt.cpu().numpy())

    metrics = {}

    metrics["accuracy"] = accuracy_score(all_labels, all_preds)
    metrics["balanced_accuracy"] = balanced_accuracy_score(all_labels, all_preds)
    metrics["f1"] = f1_score(all_labels, all_preds)

    metrics["precision_class0"] = precision_score(all_labels, all_preds, pos_label=0)
    metrics["precision_class1"] = precision_score(all_labels, all_preds, pos_label=1)

    metrics["recall_class0"] = recall_score(all_labels, all_preds, pos_label=0)
    metrics["recall_class1"] = recall_score(all_labels, all_preds, pos_label=1)

    return metrics

metrics_std  = compute_metrics(model, training_dataloader, device)
metrics_unb  = compute_metrics(model_unbalanced, training_unbalanced_dataloader, device)
metrics_hunb = compute_metrics(model_heavy_unbalanced, training_heavy_unbalanced_dataloader, device)

def print_metrics(name, m):
    print(f"\n=== {name} ===")
    print(f"Accuracy:           {m['accuracy']:.4f}")
    print(f"Balanced Accuracy:  {m['balanced_accuracy']:.4f}")
    print(f"F1-score:           {m['f1']:.4f}")
    print(f"Precision class 0:  {m['precision_class0']:.4f}")
    print(f"Precision class 1:  {m['precision_class1']:.4f}")
    print(f"Recall class 0:     {m['recall_class0']:.4f}")
    print(f"Recall class 1:     {m['recall_class1']:.4f}")

print_metrics("Standard", metrics_std)
print_metrics("Unbalanced 1%", metrics_unb)
print_metrics("Heavy Unbalanced 0.2%", metrics_hunb)

Define a dataframe and plot the predictions and GT. As expected, the predictions become blurred near the boundary.

In [ ]:
def plot_predictions_scatter(model, test_dataset, device, title):
    model.eval()
    all_preds = []
    all_labels = []
    x_vals = []
    y_vals = []

    # Ciclo sul test set
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

    with torch.no_grad():
        for xy_input, label_gt in test_loader:
            xy_input = xy_input.to(device)
            logits = model(xy_input)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_gt.cpu().numpy())
            x_vals.extend(xy_input[:,0].cpu().numpy())  # x_range
            y_vals.extend(xy_input[:,1].cpu().numpy())  # y_range

    # Crea dataframe per scatterplot
    df_plot = pd.DataFrame({
        "x_range": x_vals,
        "y_range": y_vals,
        "label": all_labels,
        "prediction": all_preds
    })

    # Scatterplot
    plt.figure(figsize=(6,6))
    sns.scatterplot(
        data=df_plot,
        x="x_range",
        y="y_range",
        hue="label",
        style="prediction",
        palette="Set1",
        alpha=0.7
    )
    plt.title(title)
    plt.show()

plot_predictions_scatter(model, training_dataset, device, "Prediction Standard Model")
plot_predictions_scatter(model_unbalanced, training_unbalanced_dataset, device, "Prediction Unbalanced 1%")
plot_predictions_scatter(model_heavy_unbalanced, training_heavy_unbalanced_dataset, device, "Prediction Heavy Unbalanced 0.2%")

## Regularization strategies
We wanto to add a Dropout regularization to improve results and avoid early convergence of the model. The next step redefines the model adding a dropout layer.

In [ ]:
# TRICK 2: Balanced weights
def create_balanced_dataloader(dataset, batch_size=10, shuffle=False):
    labels = [label for _, label in dataset]
    labels = np.array(labels)
    
    class_sample_count = np.array([len(np.where(labels==t)[0]) for t in np.unique(labels)])
    
    weight_per_class = 1. / class_sample_count
    samples_weight = np.array([weight_per_class[t] for t in labels])
    samples_weight = torch.from_numpy(samples_weight).float()
    
    sampler = WeightedRandomSampler(
        weights=samples_weight,
        num_samples=len(samples_weight),
        replacement=True
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False  # shuffle deve essere False se usi sampler
    )
    
    return dataloader


In [ ]:
# TRICK 0: Dropout regularization

# Define the model:
model_d = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

model_unbalanced_d = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

model_heavy_unbalanced_d = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Dropout(p=0.2),       # Dropout
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

# Move the model to device (GPU):
model_d.to(device)
model_unbalanced_d.to(device)
model_heavy_unbalanced_d.to(device)

# Convert weights type to float:
model_d = model_d.float()
model_unbalanced_d = model_unbalanced_d.float()
model_heavy_unbalanced_d = model_heavy_unbalanced_d.float()

# Loss function (Cross entropy loss):
loss_fn = nn.CrossEntropyLoss() # It applies log_softmax and NLL directly

# Optimizer (Adam): 
learning_rate = 1e-3 # TRICK 1: reduce learning rate (or better, adaptive learning rate)
optimizers_d = {
    "standard": optim.Adam(model_d.parameters(), lr=learning_rate),
    "unbalanced": optim.Adam(model_unbalanced_d.parameters(), lr=learning_rate),
    "heavy_unbalanced": optim.Adam(model_heavy_unbalanced_d.parameters(), lr=learning_rate)
}

# TRICK 2: INCREASED BATCH SIZE
batch_size = 100 

# TRICK 3: Shuffle == TRUE
training_dataloader_d = create_balanced_dataloader(training_dataset, batch_size=batch_size, shuffle=True)
training_unbalanced_dataloader_d = create_balanced_dataloader(training_unbalanced_dataset, batch_size=batch_size, shuffle=True)
training_heavy_unbalanced_dataloader_d = create_balanced_dataloader(training_heavy_unbalanced_dataset, batch_size=batch_size, shuffle=True)


# Model info
models_info_d = [
    {
        "name": "standard",
        "model": model_d,
        "dataloader": training_dataloader_d  # training standard
    },
    {
        "name": "unbalanced",
        "model": model_unbalanced_d,
        "dataloader": training_unbalanced_dataloader_d  # training 1% class 1
    },
    {
        "name": "heavy_unbalanced",
        "model": model_heavy_unbalanced_d,
        "dataloader": training_heavy_unbalanced_dataloader_d  # training 0.2% class 1
    }
]



# Number of training epochs:
n_epochs = 50

all_training_losses = {}
all_validation_losses  = {}

for info in models_info_d:
    print(f"\nTraining model: {info['name']}")
    
    current_model = info['model']
    current_dataloader = info['dataloader']
    current_optimizer = optimizers_d[info['name']]

    training_loss_list = []
    validation_loss_list = []
    
    # Iterate over the epochs:
    for epoch in range(n_epochs):
    
        # Set the model to Training mode: This interacts with certain kind of network layers (such as Dropout layers)
        current_model.train()  
    
        # Temporary variable to store the loss on the whole epoch as a convergence metric
        running_loss = 0.0
    
        # Iterate on the whole dataset using the dataloader.
        for xy_input, label_gt in current_dataloader:
            # Load inputs ad move to device (GPU)
            xy_input, label_gt = xy_input.to(device), label_gt.to(device)
    
            # Clear previous gradients
            current_optimizer.zero_grad()  
            
            # Forward pass (model calls)
            logits_model = current_model(xy_input)  
    
            # Compute loss (supervised case)
            loss = loss_fn(logits_model, label_gt)  
            
            # Backpropagation 
            loss.backward()  
    
            # Update parameters (optimization step)
            current_optimizer.step()  
    
            # Update running loss as convergence metric
            running_loss += loss.item()
            
        ## Calculate loss on validation as an additional metric to evaluate overfitting
        # Set the model to Evaluation mode:
        current_model.eval()
    
        # Temporary variable to store the validation loss:
        running_val_loss = 0.0
        
        # Deactivate gradient computation
        with torch.no_grad():
            for xy_input, label_gt in validation_dataloader:
                # Load inputs ad move to device (GPU)
                xy_input, label_gt = xy_input.to(device), label_gt.to(device)
    
                # Forward pass (model calls)
                logits_model = current_model(xy_input)
    
                # Compute loss (supervised case)
                loss = loss_fn(logits_model, label_gt)
    
                # Update validation running loss as convergence metric
                running_val_loss += loss.item()
                
        # Average epoch loss
        epoch_training_loss = running_loss / len(current_dataloader)
        epoch_validation_loss = running_val_loss / len(validation_dataloader)
    
        # Append the losses to the list:
        training_loss_list.append(epoch_training_loss)
        validation_loss_list.append(epoch_validation_loss)
                
    
        # Convergence metric
        print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_training_loss:.4f}, Val. Loss: {epoch_validation_loss:.4f}")
        
    # Salva i risultati nel dizionario
    all_training_losses[info['name']] = training_loss_list
    all_validation_losses[info['name']] = validation_loss_list

In [ ]:
y_true_std_d, y_pred_std_d = get_predictions(model_d, training_dataloader_d, device)
y_true_unb_d, y_pred_unb_d = get_predictions(model_unbalanced_d, training_dataloader_d, device)
y_true_hunb_d, y_pred_hunb_d = get_predictions(model_heavy_unbalanced_d, training_dataloader_d, device)

cm_std_d  = normalize_cm(confusion_matrix(y_true_std_d, y_pred_std_d))
cm_unb_d  = normalize_cm(confusion_matrix(y_true_unb_d, y_pred_unb_d))
cm_hunb_d = normalize_cm(confusion_matrix(y_true_hunb_d, y_pred_hunb_d))

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18,5))

cms = [cm_std_d, cm_unb_d, cm_hunb_d]
titles = ["Standard _d", "Unbalanced _d (1%)", "Heavy Unbalanced _d (0.2%)"]

for ax, cm, title in zip(axes, cms, titles):
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        vmin=0,
        vmax=1,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

Plot of all the trained models

In [ ]:
plot_predictions_scatter(model_d, training_dataset, device, "Prediction Standard Model")
plot_predictions_scatter(model_unbalanced_d, training_unbalanced_dataset, device, "Prediction Unbalanced 1%")
plot_predictions_scatter(model_heavy_unbalanced_d, training_heavy_unbalanced_dataset, device, "Prediction Heavy Unbalanced 0.2%")

Plot on test set of the most unbalanced case

In [ ]:
plot_predictions_scatter(model_heavy_unbalanced_d, test_dataset, device, "Prediction Heavy Unbalanced 0.2%")